## ESM for DMS - Exploratory Data Analysis

### Introduction and Data

This notebook focuses on the exploration of DMS datasets, with the goal being to look into patterns in mutations across positions and scoring. This is an important step before building any imputation pipelines - e.g. if we are able to determine trends based off just data visualisation and other classical techniques then we can avoid redundancy in looking at more complex methods later.

Dataset files per protein required:
- **wild_type.fasta** - the unmutated wild-type protein sequence
- **all_variants.fasta** - every mutant sequence, which will be utilised later and fed into ESM to generate embeddings for score predictions
- **scores.csv** - the lab measured functional score for explored variants, e.g. variant T2A and score X means wild-type T at position 2 mutated to A has functional score X

In [ ]:
# Importing necessary libraries and functions
import requests # HTTP requests
import py3Dmol # 3D molecule viewer
import pandas as pd # Data manipulation
import matplotlib.pyplot as plt # Plotting
import seaborn as sns # Plotting
from Bio import SeqIO # Parsing FASTA files
from Bio.Data.IUPACData import protein_letters # Amino acid letters

# Importing from created .py files
import data_loader # Loads data
import informatics # Plotting functions
import sequence_function_heatmap # Plots sequence-function heatmaps

### Loading Datasets and Defining Explorative Functions

This section focuses on loading the data, initial checks, and defining functions which will effectively load the data based on the specified protein. The Biopython package is used to import protein letters and also to parse the wild-type FASTA files efficiently. The functions following from the loader enable data summaries, score plotting, and sequence-function heatmaps to be generated easily.

In [ ]:
# Example
datasets = ["HRAS", "P53", "PTEN"] # Setting datasets to analyse
amino_acids = list(protein_letters) # Standard 20 amino acids

In [ ]:
# Validating all_variants.fasta matches the expected mutational space for each dataset
for dataset in datasets:
    records = SeqIO.parse(f"datasets/{dataset}/all_variants.fasta", "fasta") # Parsing all variants
    all_variant_names = set(record.id for record in records) # Storing variant names in a set
    wild_type, scores_df = data_loader.loadDataset(dataset, verbose=False) # Loading wild type and scores

    expected_variants = set() # Initialising empty set to store expected variants

    # Adding expected variants based off possible amino acid substitutes
    for position in range(1, len(wild_type) + 1):
        wild_type_residue = wild_type[position - 1]
        for mutant_residue in amino_acids:
            if mutant_residue != wild_type_residue:
                expected_variants.add(f"{wild_type_residue}{position}{mutant_residue}") # Adding variant

    print(f"{dataset} matches expected set:", expected_variants == all_variant_names) # Verifying set equality

This validates that all_variants.fasta only contains feasible variants, but we need to ensure that there are no duplicate variants (since sets don't store duplicates), and that the number of total variants is as expected. This can be done simply by checking for duplicate events - if the sets match then all_variants.fasta only contains expected variants, so if we check for duplicates we of course validate that there are no duplicates but we also validate that the number of variants is correct because the only way it would be incorrect is if it contained a duplicate variant.

In [ ]:
# Validating all_variants.fasta has no duplicate variants for each dataset
for dataset in datasets:
    raw_variant_names = [] # Initialising list for variant names
    for record in SeqIO.parse(f"datasets/{dataset}/all_variants.fasta", "fasta"):
        raw_variant_names.append(record.id)

    duplicate_count = len(raw_variant_names) - len(set(raw_variant_names)) # Comparing raw count to unique count
    print(f"{dataset}: {duplicate_count} duplicates found")

### Running Functions and Examining Outputs

In [ ]:
# Running EDA for each dataset
summaries = {}

for dataset in datasets:
    wild_type, scores_df = data_loader.loadDataset(dataset)
    informatics.printSummary(dataset, wild_type, scores_df)
    informatics.printOutliers(dataset, scores_df)
    informatics.plotSubstitutionHeatmap(dataset, scores_df)
    informatics.plotScores(dataset, scores_df)
    informatics.plotStandardDeviations(dataset, scores_df)
    sequence_function_heatmap.plotHeatmap(f"{dataset} Sequence-Function Heat-Map", wild_type, scores_df)
    
    summaries[dataset] = {
        "protein_length": len(wild_type),
        "coverage": 100 * len(scores_df) / (len(wild_type) * 19),
        "score_mean": scores_df["score"].mean(),
        "score_std": scores_df["score"].std(),
    }